<a href="https://colab.research.google.com/github/ncz-cruz/collab-test/blob/main/Wan2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -r requirements.txt
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 2. Install WanVideoWrapper custom nodes & VHS
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/kijai/ComfyUI-WanVideoWrapper.git
%cd /content/ComfyUI/custom_nodes/ComfyUI-WanVideoWrapper
!pip install -r requirements.txt

%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
%cd /content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite
!pip install -r requirements.txt

# 3. Download model weights
import os

models_dir = "/content/ComfyUI/models/diffusion_models"
os.makedirs(models_dir, exist_ok=True)

# Wan 2.1 T2V 1.3B (bf16)
!wget -q --show-progress -O {models_dir}/wan2.1_t2v_1.3B_bf16.safetensors \
    "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/diffusion_models/wan2.1_t2v_1.3B_bf16.safetensors"

text_enc_dir = "/content/ComfyUI/models/text_encoders"
os.makedirs(text_enc_dir, exist_ok=True)
!wget -q --show-progress -O {text_enc_dir}/umt5_xxl_fp8_e4m3fn_scaled.safetensors \
    "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors"

vae_dir = "/content/ComfyUI/models/vae"
os.makedirs(vae_dir, exist_ok=True)
!wget -q --show-progress -O {vae_dir}/wan_2.1_vae.safetensors \
    "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors"

print("✓ All model weights downloaded.")

# 4. Install cloudflared
%cd /content
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# 5. Start ComfyUI + Tunnel & Keep Alive
import subprocess
import time
import re
import requests
from IPython.display import clear_output

# Start ComfyUI in background
comfy_proc = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd="/content/ComfyUI",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

time.sleep(15)
print("ComfyUI starting...")

# Start cloudflared tunnel
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

time.sleep(10)

for _ in range(30):
    try:
        r = requests.get("http://localhost:8188/system_stats", timeout=5)
        if r.status_code == 200:
            print("✓ ComfyUI is running!")
            break
    except:
        time.sleep(2)

output = tunnel_proc.stdout.read1(4096).decode()
urls = re.findall(r'https://[\w-]+\.trycloudflare\.com', output)

tunnel_url = urls[0] if urls else None

while True:
    clear_output(wait=True)
    if tunnel_url:
        print(f"\n{'='*60}")
        print(f"  COMFYUI TUNNEL URL: {tunnel_url}")
        print(f"{'='*60}")
        print(f"\nCopy this URL to your .env file:")
        print(f'  COMFYUI_URL="{tunnel_url}"\n')
    else:
        print("⚠ Could not detect tunnel URL automatically.")

    try:
        r = requests.get("http://localhost:8188/system_stats", timeout=5)
        stats = r.json()
        devices = stats.get("devices", [{}])
        gpu = devices[0] if devices else {}
        vram_used = gpu.get("vram_total", 0) - gpu.get("vram_free", 0)
        vram_total = gpu.get("vram_total", 0)
        print(f"[{time.strftime('%H:%M:%S')}] ComfyUI alive | "
              f"VRAM: {vram_used // (1024**3)}GB / {vram_total // (1024**3)}GB")
    except Exception as e:
        print(f"[{time.strftime('%H:%M:%S')}] ComfyUI check: {e}")
    time.sleep(60)